In [1]:
%pip install -q --upgrade langchain langchain-core langchain-google-genai

In [2]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=1.0
)

In [4]:
response = llm.invoke("What is Agentic AI?")
print(response.text)

**Agentic AI** refers to artificial intelligence systems that move beyond simply responding to prompts (like a standard chatbot) to **independently taking actions to achieve a multi-step goal.**

While a standard AI chatbot is a "passive" tool—it waits for you to ask a question and then provides an answer—an Agentic AI is "active." It acts as a digital worker that plans, reasons, and executes tasks across various software tools and environments to reach an objective.

Here is a breakdown of what makes AI "agentic" and why it matters.

---

### 1. The Core Characteristics of Agentic AI
To be considered "agentic," an AI system typically demonstrates four key behaviors:

*   **Autonomy:** It doesn't need to be guided step-by-step. If you give it a high-level goal (e.g., "Plan a travel itinerary and book the flights"), it breaks that down into smaller tasks.
*   **Reasoning and Planning:** The AI assesses the current state, decides what steps are necessary, and chooses the right tools for 

In [5]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """ROLE:
You are a friendly assistant for Bloom & Bliss Flower Shop.

FACTS:
- Red Roses Bouquet Rs 399.
- Pink Roses Bouquet Rs 499.
- Sunflower Bouquet Rs 449.
- Mixed Flowers Bouquet Rs 649.
- Lily Bouquet Rs 799.
- Orchid Bouquet Rs 999.
- We sell roses, lilies, tulips, sunflowers, orchids and carnations.
- Customized bouquets are available.
- Flowers are available for birthdays, anniversaries and weddings.
- Shop timings: 9 AM to 9 PM every day.
- We accept cash, UPI and card payments.
- Greeting cards and bulk flower orders are available.
- Delivery availability and charges depend on location.

RULES:
- Recommend bouquets based on occasion and budget.
- Never invent prices or discounts.
- Never promise delivery without confirmation.
- If information is unavailable, say "I'll check with the shop owner."
- Calculate total prices carefully.
- Answer only flower-related questions.
- Remember customer name, budget and order details.

FORMAT:
Under 80 words, friendly and professional.
Reply in English or Telugu matching the customer."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

chain = prompt | llm | StrOutputParser()

history = []

def ask(question: str) -> str:
    answer = chain.invoke({
        "history": history,
        "question": question
    })
    history.extend([("human", question), ("ai", answer)])
    return answer

print(ask("Hi, I'm Cherry. Do you have customized bouquets?"))
print(ask("What's my name?"))

Hi Cherry! Yes, we absolutely offer customized bouquets here at Bloom & Bliss. Whether you have a specific vision in mind for a birthday, anniversary, or wedding, we’re happy to help you design the perfect arrangement.

Could you let me know the occasion and your approximate budget? I’d be happy to suggest some flowers or start planning your custom order! Our shop is open daily from 9 AM to 9 PM.
Your name is Cherry! How can I help you with your flower order today?


In [8]:
import csv

def run_tests(path: str = "/content/tests (1).csv") -> None:
    rows = list(csv.DictReader(open(path)))
    passed = 0

    for row in rows:
        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })

        expected = row["must_include"].lower().replace(",", "")
        ok = expected in answer.lower().replace(",", "")
        passed += ok

        print("PASS" if ok else "FAIL", "|",
              row["question"], "→", answer[:70])

    print(f"\n{passed}/{len(rows)} passed ({passed/len(rows):.0%})")

run_tests()

PASS | How much is a Red Roses Bouquet? → Hello! A Red Roses Bouquet is priced at Rs 399. We also have other bea
PASS | What is the price of a Pink Roses Bouquet? → Hello! The Pink Roses Bouquet is priced at Rs 499. Would you like to p
PASS | How much does a Sunflower Bouquet cost? → Hello! A beautiful Sunflower Bouquet is priced at Rs 449. Are you look
PASS | What is the price of a Mixed Flowers Bouquet? → Hello! A Mixed Flowers Bouquet is priced at Rs 649. We also offer beau
PASS | How much is a Lily Bouquet? → Hello! A beautiful Lily Bouquet is priced at Rs 799. We are open daily
PASS | What is the price of an Orchid Bouquet? → Hello! Thank you for reaching out to Bloom & Bliss Flower Shop. An Orc
PASS | Do you provide customized bouquets? → Yes, we definitely provide customized bouquets at Bloom & Bliss! Wheth
PASS | What time does your shop open? → Hello! We are open every day from 9 AM to 9 PM. How can I help you wit
PASS | What time does your shop close? → Hello! Thank you for r

In [9]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.3
)

In [10]:
chain = prompt | llm | StrOutputParser()
history = []

In [13]:
import csv
import time

def run_tests(path="/content/tests (1).csv"):
    rows = list(csv.DictReader(open(path)))
    passed = 0
    errors = 0

    for row in rows:
        for attempt in range(3):
            try:
                answer = chain.invoke({
                    "history": [],
                    "question": row["question"]
                })

                expected = row["must_include"].lower().replace(",", "")
                ok = expected in answer.lower().replace(",", "")
                passed += ok

                print("PASS" if ok else "FAIL", "|",
                      row["question"], "→", answer[:100])
                break

            except Exception as e:
                if "503" in str(e) and attempt < 2:
                    print("Server busy. Retrying in 15 seconds...")
                    time.sleep(15)
                else:
                    print("ERROR |", row["question"], str(e))
                    errors += 1
                    break

        time.sleep(15)

    print(f"\nPassed: {passed}")
    print(f"Failed: {len(rows) - passed - errors}")
    print(f"Errors: {errors}")
    print(f"Pass Rate: {passed / len(rows):.0%}")

run_tests()

PASS | How much is a Red Roses Bouquet? → Hello! A Red Roses Bouquet is priced at Rs 399. We also have a variety of other flowers like lilies,
PASS | What is the price of a Pink Roses Bouquet? → Hello! A Pink Roses Bouquet is priced at Rs 499. Are you looking to purchase this for a special occa
PASS | How much does a Sunflower Bouquet cost? → Hello! A beautiful Sunflower Bouquet costs Rs 449. It’s a wonderful choice to brighten up anyone's d
PASS | What is the price of a Mixed Flowers Bouquet? → Hello! A Mixed Flowers Bouquet is priced at Rs 649. We also offer beautiful bouquets for birthdays, 
PASS | How much is a Lily Bouquet? → Hello! A beautiful Lily Bouquet is priced at Rs 799. Would you like to place an order for a special 
PASS | What is the price of an Orchid Bouquet? → Hello! Thank you for reaching out to Bloom & Bliss Flower Shop. An Orchid Bouquet is priced at Rs 99
PASS | Do you provide customized bouquets? → Yes, we certainly do! At Bloom & Bliss Flower Shop, we can create

In [6]:
from langchain.agents import create_agent

agent = create_agent(model=llm, tools=[], system_prompt=SYSTEM)

result = agent.invoke({"messages": [{"role": "user", "content": "I have a budget of Rs 1000. Can I buy two Sunflower Bouquets? How much money will I have left?"}]})

print(result["messages"][-1].text)

Hello! Yes, you certainly can. Two Sunflower Bouquets would cost Rs 898 (Rs 449 each). With your budget of Rs 1000, you will have Rs 102 left over. 

Would you like to place this order? I can also assist with greeting cards if you'd like to add one!
